# 二阶段分类数据集构建

这个 notebook 直接从标准 LabelMe 标注或 `web_ui` 导出的标注生成二阶段分类数据集。

它会生成两版数据：
- `standard`: 按原始 bbox 裁剪
- `tight`: 在 bbox 内部自动收紧内容后再裁剪


## 0. 依赖导入


In [ ]:
from pathlib import Path
import json
from collections import Counter, defaultdict

import numpy as np
from PIL import Image

print('✓ 依赖导入完成')


## 1. 核心函数


In [ ]:
def _normalize_shape_bbox(points):
    if not points or len(points) < 2:
        return None

    x_values = [point[0] for point in points]
    y_values = [point[1] for point in points]
    return [
        int(min(x_values)),
        int(min(y_values)),
        int(max(x_values)),
        int(max(y_values)),
    ]


def load_annotations(annotations_dir):
    annotations = []
    for annotation_path in sorted(Path(annotations_dir).glob('*.json')):
        payload = json.loads(annotation_path.read_text(encoding='utf-8'))
        metadata = payload.get('metadata', {})
        original_path = metadata.get('original_path') or payload.get('imagePath')
        instances = metadata.get('instances', [])

        if instances:
            for instance in instances:
                annotations.append({
                    **instance,
                    'annotation_path': str(annotation_path),
                    'original_path': instance.get('original_path') or original_path,
                })
            continue

        for index, shape in enumerate(payload.get('shapes', []), start=1):
            if shape.get('shape_type', 'rectangle') != 'rectangle':
                continue
            bbox = _normalize_shape_bbox(shape.get('points', []))
            if not bbox:
                continue
            annotations.append({
                'id': shape.get('id', index),
                'class_name': shape.get('label'),
                'bbox': bbox,
                'annotation_path': str(annotation_path),
                'original_path': original_path,
            })

    return annotations


def _resolve_image_path(source_images_dir, original_path):
    original_path = Path(original_path)
    candidate = Path(source_images_dir) / original_path
    if candidate.exists():
        return candidate
    return Path(source_images_dir) / original_path.name


def _apply_padding(bbox, width, height, padding_ratio):
    x1, y1, x2, y2 = bbox
    box_width = max(x2 - x1, 1)
    box_height = max(y2 - y1, 1)
    pad_x = box_width * padding_ratio
    pad_y = box_height * padding_ratio
    return [
        max(int(x1 - pad_x), 0),
        max(int(y1 - pad_y), 0),
        min(int(x2 + pad_x), width),
        min(int(y2 + pad_y), height),
    ]


def tighten_bbox_to_content(image, bbox, min_content_size=16, threshold=245):
    x1, y1, x2, y2 = [int(value) for value in bbox]
    crop = image.crop((x1, y1, x2, y2)).convert('L')
    crop_array = np.array(crop)
    foreground_mask = crop_array < threshold

    if not foreground_mask.any():
        return [x1, y1, x2, y2]

    ys, xs = np.where(foreground_mask)
    content_x1 = x1 + int(xs.min())
    content_y1 = y1 + int(ys.min())
    content_x2 = x1 + int(xs.max()) + 1
    content_y2 = y1 + int(ys.max()) + 1

    if (content_x2 - content_x1) < min_content_size or (content_y2 - content_y1) < min_content_size:
        return [x1, y1, x2, y2]

    return [content_x1, content_y1, content_x2, content_y2]


def _pick_split(original_path, split_ratios):
    train_ratio, val_ratio, _ = split_ratios
    bucket = sum(ord(char) for char in str(original_path)) % 1000 / 1000.0
    if bucket < train_ratio:
        return 'train'
    if bucket < train_ratio + val_ratio:
        return 'val'
    return 'test'


def build_dataset(
    annotations_dir,
    source_images_dir,
    output_dir,
    split_ratios=(0.8, 0.1, 0.1),
    padding_ratio=0.05,
    min_crop_size=16,
    tighten_content=False,
):
    annotations = load_annotations(annotations_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    class_counts = Counter()
    skip_reasons = Counter()
    saved_instances = 0
    grouped_annotations = defaultdict(list)

    for instance in annotations:
        original_path = instance.get('original_path')
        if not original_path:
            skip_reasons['invalid_instance'] += 1
            continue
        grouped_annotations[original_path].append(instance)

    for original_path, instances in grouped_annotations.items():
        image_path = _resolve_image_path(source_images_dir, original_path)
        if not image_path.exists():
            skip_reasons['missing_image'] += len(instances)
            continue

        with Image.open(image_path) as image:
            rgb_image = image.convert('RGB')
            for instance in instances:
                label = instance.get('class_name')
                bbox = instance.get('bbox')

                if not label or not bbox:
                    skip_reasons['invalid_instance'] += 1
                    continue

                effective_bbox = tighten_bbox_to_content(rgb_image, bbox, min_content_size=min_crop_size) if tighten_content else bbox
                padded_bbox = _apply_padding(effective_bbox, rgb_image.width, rgb_image.height, padding_ratio)
                x1, y1, x2, y2 = padded_bbox

                if (x2 - x1) < min_crop_size or (y2 - y1) < min_crop_size:
                    skip_reasons['crop_too_small'] += 1
                    continue

                split_name = _pick_split(original_path, split_ratios)
                target_dir = output_dir / split_name / label
                target_dir.mkdir(parents=True, exist_ok=True)

                crop = rgb_image.crop((x1, y1, x2, y2))
                output_name = f"{Path(original_path).stem}_inst{instance.get('id', saved_instances + 1)}.jpg"
                crop.save(target_dir / output_name, quality=95)

                saved_instances += 1
                class_counts[label] += 1

    summary = {
        'total_instances': len(annotations),
        'saved_instances': saved_instances,
        'skipped_instances': len(annotations) - saved_instances,
        'class_counts': dict(class_counts),
        'skip_reasons': dict(skip_reasons),
    }

    summary_path = output_dir / 'dataset_summary.json'
    summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    return summary


print('✓ 核心函数准备完成')


## 2. 路径与参数配置


In [ ]:
ANNOTATIONS_DIR = Path('/home/hujing/yolo-web-ui/offline_workspace/datasets_annotated_2class')
SOURCE_IMAGES_DIR = Path('/home/hujing/yolo-web-ui/offline_workspace/datasets_annotated_2class')
OUTPUT_ROOT = Path('/home/hujing/yolo-web-ui/offline_workspace/datasets')

STANDARD_OUTPUT_DIR = OUTPUT_ROOT / 'insulator_cls_dataset_standard'
TIGHT_OUTPUT_DIR = OUTPUT_ROOT / 'insulator_cls_dataset_tight'

MIN_CROP_SIZE = 24
STANDARD_PADDING_RATIO = 0.08
TIGHT_PADDING_RATIO = 0.02

print('ANNOTATIONS_DIR =', ANNOTATIONS_DIR)
print('SOURCE_IMAGES_DIR =', SOURCE_IMAGES_DIR)
print('STANDARD_OUTPUT_DIR =', STANDARD_OUTPUT_DIR)
print('TIGHT_OUTPUT_DIR =', TIGHT_OUTPUT_DIR)


## 3. 快速检查目录


In [ ]:
json_count = len(list(ANNOTATIONS_DIR.glob('*.json')))
image_count = len([p for p in SOURCE_IMAGES_DIR.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}])
print('json_count =', json_count)
print('image_count =', image_count)
print('预览前 3 个实例:')
print(load_annotations(ANNOTATIONS_DIR)[:3])


## 4. 生成 standard 数据集


In [ ]:
standard_summary = build_dataset(
    annotations_dir=ANNOTATIONS_DIR,
    source_images_dir=SOURCE_IMAGES_DIR,
    output_dir=STANDARD_OUTPUT_DIR,
    padding_ratio=STANDARD_PADDING_RATIO,
    min_crop_size=MIN_CROP_SIZE,
    tighten_content=False,
)

print(json.dumps(standard_summary, indent=2, ensure_ascii=False))


## 5. 生成 tight 数据集


In [ ]:
tight_summary = build_dataset(
    annotations_dir=ANNOTATIONS_DIR,
    source_images_dir=SOURCE_IMAGES_DIR,
    output_dir=TIGHT_OUTPUT_DIR,
    padding_ratio=TIGHT_PADDING_RATIO,
    min_crop_size=MIN_CROP_SIZE,
    tighten_content=True,
)

print(json.dumps(tight_summary, indent=2, ensure_ascii=False))


## 6. 输出目录提示


In [ ]:
print('standard 输出目录:')
print(STANDARD_OUTPUT_DIR)
print('\ntight 输出目录:')
print(TIGHT_OUTPUT_DIR)
print('\n你可以重点对比以下目录:')
print(STANDARD_OUTPUT_DIR / 'train' / 'abnormal')
print(TIGHT_OUTPUT_DIR / 'train' / 'abnormal')
